In [1]:
import warnings
warnings.filterwarnings(action="ignore")
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [7]:
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

<a name="ComputationGraph"></a>
# 计算图框架

QuantStudio 系统以计算图为核心, 系统中主要的计算以有向图的形式表达，图由若干个节点组成，每个节点完成某种定义的计算，节点之间的依赖关系以有向边来表示，边从依赖节点指向被依赖的节点。计算引擎负责调度和执行计算图。

![QuantStudio系统](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/QuantStudio系统.jpg)

每个计算节点必须继承自 `QuantStudio.Core.Node.Node`，对象创建的 `__init__` 方法除了 QuantStudio 对象的三个输入参数外还有一个 deps 参数，其为当前节点依赖的节点列表。

要实现一个计算节点，必须实现 Node 的五个方法: `init_compute`, `prepare_compute`, `forward_compute`, `backward_compute`, `merge_result`, 其中除 `backward_compute` 外均有默认实现，`backward_compute` 是节点运算的主逻辑实现。

In [3]:
from QuantStudio.Core.Node import Node
print(qs_help(Node.init_compute))
print("-" * 10)
print(qs_help(Node.prepare_compute))
print("-" * 10)
print(qs_help(Node.forward_compute))
print("-" * 10)
print(qs_help(Node.backward_compute))
print("-" * 10)
print(qs_help(Node.merge_result))

类型: function
模块: QuantStudio.Core.Node
签名: Node.init_compute(self, path: List[str], init_data: Any, context: QuantStudio.Core.Node.Context) -> List[Any]
说明文档:
    按照边的方向传递数据执行初始化，可以修改 context 中的全局变量，最好不要有耗时的计算
    
    Args:
        path: 运行至当前节点的路径, 由路径上所有节点 ID 组成的 list
        init_data: 上游传递的数据
        context: 全局上下文对象
    
    Returns:
        产生的向下游传递的数据列表, 如果返回空 list 表示终止继续向下的初始化
----------
类型: function
模块: QuantStudio.Core.Node
签名: Node.prepare_compute(self, prepare_data: Any, context: QuantStudio.Core.Node.Context)
说明文档:
    主逻辑计算开始前的准备计算, 不可以修改 context 中的全局变量，最好将 IO 操作在这里实现，只对 context.PrepareNodeDict 中的节点执行该操作
    
    Args:
        prepare_data: 执行准备计算所需的数据, 在 init_compute 时生成在 context.PrepareNodeDict 中
        context: 全局上下文对象
----------
类型: function
模块: QuantStudio.Core.Node
签名: Node.forward_compute(self, path: List[str], fwd_data: Any, context: QuantStudio.Core.Node.Context) -> Tuple[List[Any], Any]
说明文档:
    按照边的方向传递数据执行运算, 即从父节点向子节点传递
    
    Args:
        path: 运行至当前节点

节点计算方法的入参里有一个全局上下文对象 Context，其继承自 `__QS_Args__`，主要用于维护全局信息以及节点的运行时状态，其字段如下：

In [8]:
from QuantStudio.Core.Node import Context

DemoContext = Context()
display(Markdown(DemoContext.info()))

* Mode(运行模式): typing.Literal['PRD', 'DEBUG'], 默认值 'PRD', 当前取值: 'PRD'
* NodeDict(节点集): typing.Dict[str, QuantStudio.Core.Node.Node], 默认值 {}, {节点ID: Node}, 本次运算的所有 Node, 由计算引擎生成, 当前取值: {}
* NodeState(节点状态): typing.Dict[str, typing.Any], 默认值 {}, {节点ID: Any}, 运算中用于存储节点的临时数据，由节点生成和维护, 当前取值: {}
* PrepareNodeDict(准备节点列表): typing.Dict[str, typing.Tuple[str, typing.Any]], 默认值 {}, {准备ID: (节点ID, Any)}, 需要执行准备操作的节点列表, 当前取值: {}
* PID(当前进程ID): <class 'str'>, 默认值 '0', 当前的运行进程 ID, 默认为 '0', 当前取值: '0'
* PIDList(全部进程ID): typing.List[str], 默认值 ['0'], 所有运行进程 ID 列表, 当前取值: ['0']
* SplitType(切分方式): typing.Literal['连续切分', '间隔切分'], 默认值 '连续切分', 当前取值: '连续切分'
* Event(同步Event): <class 'dict'>, 默认值 {}, {节点ID: Event}, 用于多进程同步的 Event 数据, 当前取值: {}
* Sub2MainQueue: typing.Optional[typing.Any], 默认值 None, 用于子进程向主进程发送消息, 当前取值: None
* TaskExecutor(并行执行器): typing.Optional[concurrent.futures._base.Executor], 默认值 None, 给到节点用于并行计算, 当前取值: None
* MaxWorkers(最大并行数量): <class 'int'>, 默认值 1, 节点执行并行计算的最大并发量, 当前取值: 1
* DataCache(数据缓存): typing.Optional[QuantStudio.Core.Cache.Cache], 默认值 None, 当前取值: None
* ExtraData(其他数据): <class 'dict'>, 默认值 {}, 当前取值: {}

计算图的调度和运行由计算引擎对象 Engine 执行，每种计算引擎的实现均继承自 `QuantStudio.Core.CalcEngine.Engine`，其执行计算的主要方法是 `run`

In [9]:
from QuantStudio.Core.CalcEngine import Engine

print(qs_help(Engine.run))

类型: function
模块: QuantStudio.Core.CalcEngine
签名: Engine.run(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None, fwd_data_list: Optional[List[Any]] = None) -> List[Any]
说明文档:
    给定节点列表, 执行所有节点的计算, 返回每个节点的计算结果
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文对象
        init_data_list: 初始化数据列表
        fwd_data_list: 前向计算输入数据列表
    
    Returns:
        节点计算的结果列表


In [10]:
# 使用计算图框架实现四则运算
from typing import Any, List, Optional

import numpy as np
import pandas as pd

from QuantStudio.Core.Node import Node, Context
from QuantStudio.Core.CalcEngine import Engine

class Num(Node):
    """数字"""
    def __init__(self, value:float, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        if "Name" not in args: args = args | {"Name": str(value)}
        self._Value = value
        return super().__init__(deps=deps, args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return self._Value

class Sum(Node):
    """加法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "sum"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.sum(bwd_data_list)

class Sub(Node):
    """减法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "sub"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] - bwd_data_list[1]

class Prod(Node):
    """乘法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "prod"} | args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.prod(bwd_data_list)

class Div(Node):
    """除法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "div"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] / bwd_data_list[1]


Node1 = Prod([Sum([Num(1), Num(2)]), Num(3)], args={"Name": "(1 + 2) * 3"})
Node2 = Sum([Num(3), Prod([Num(3), Num(2)])], args={"Name": "3 + 3 * 2"})

Engine = Engine()
NodeList = [Node1, Node2]
Rslt = Engine.run(NodeList, Context())
for i, iNode in enumerate(NodeList):
    print(iNode.Name, "=", Rslt[i])

(1 + 2) * 3 = 9
3 + 3 * 2 = 9


In [ ]:
# %pip install mermaid-python

In [12]:
# 计算图的可视化
from mermaid import Mermaid
from QuantStudio.Tools.Visualization import node2dict, dict2mermaid

NodeDict = node2dict([Node1, Node2])
NodeMermaid = dict2mermaid(NodeDict)
display(Mermaid(NodeMermaid))